In [ ]:
# Phase 6 prep v2: turn raw DICOM studies into one slot-keyed .npz each.
# Three changes from Phase 2's artifact, all of which the 2026-09-05 recon named
# as most of the distance to the public 0.93 shelf:
#   * pixels cropped to a fixed PHYSICAL extent (CROP_MM), so mm/px is constant
#     across the corpus instead of an accident of acquisition. The old 256px
#     letterbox normalised no scale at all -- studies differ several-fold.
#   * series assigned to named SLOTS (plane x weighting) with no substitution,
#     so an absent view is absent and the model gets an honest presence mask.
#   * slices stored as contiguous GROUPS of 3, which become an encoder input's
#     three channels rather than one slice replicated three times.
# CPU-only, internet off. Does not touch the GPU quota the Phase 6 screens need.
import glob, os, shutil, sys, time

GIT_SHA = '801df73-wip'

src_candidates = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)
comp_candidates = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)
SRC = src_candidates[0]
COMP_DIR = comp_candidates[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

from knee.dicom import SLICE_GROUP, SLOTS, StudyDecodeError
from knee.prep import prep_slots, save_study_npz
print('knee package imported from', PKG, '| GIT_SHA', GIT_SHA)
print('slots:', [name for name, _, _ in SLOTS])

In [ ]:
# Sharding is by contiguous index range over sorted study UIDs, and it is the
# resume mechanism: /kaggle/working starts empty every run, so a lost shard
# re-runs alone rather than the whole corpus.
PILOT_N = None        # 50 for a pilot; None for the full run
SHARD_INDEX = 2       # per-shard kernels differ only in this line
N_SHARDS = 4

# 5 groups x 3 adjacent slices = 15 slices per filled slot. Sized locally before
# launching: at 336px this is 9.5 GB over the corpus (2.4 GB/shard) given the
# measured mean of 4.84 filled slots per study -- inside the output cap with
# room. If it has to come down the knob is N_GROUPS, never OUT_SIZE: resolution
# is the axis under test and 224px would miss a 1mm tear at this crop.
N_GROUPS = 5
CROP_MM = 130.0
OUT_SIZE = 336

OUT_DIR = '/kaggle/working/prepped'
os.makedirs(OUT_DIR, exist_ok=True)

import math
import numpy as np
import pandas as pd

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
series_df = pd.read_csv(f'{COMP_DIR}/train_series.csv')

gold_cols = [c for c in train_df.columns if c not in ('StudyInstanceUID', 'Report')]
gold_uids = set(train_df.loc[train_df[gold_cols].notna().any(axis=1), 'StudyInstanceUID'])

TRAIN_SERIES_DIR = f'{COMP_DIR}/train_series'
all_study_uids = sorted(
    d for d in os.listdir(TRAIN_SERIES_DIR)
    if os.path.isdir(os.path.join(TRAIN_SERIES_DIR, d))
)

per_shard = math.ceil(len(all_study_uids) / N_SHARDS)
lo = SHARD_INDEX * per_shard
study_uids = all_study_uids[lo:lo + per_shard]

if PILOT_N is not None:
    stride = max(len(study_uids) // PILOT_N, 1)
    sampled = study_uids[::stride][:PILOT_N]
    study_uids = sorted(set(sampled) | {u for u in study_uids if u in gold_uids})

print(f'{len(all_study_uids)} studies on disk, {len(gold_uids)} gold-labeled')
print(f'shard {SHARD_INDEX}/{N_SHARDS}: [{lo}, {lo + per_shard}) -> {len(study_uids)} studies '
      f'({sum(u in gold_uids for u in study_uids)} gold)')

In [ ]:
# The prep loop. Counters stay on for the full run: they cost nothing and turn
# the corpus-wide census into a free byproduct.
meta_rows = []
failed_studies = []
t0 = time.time()

for i, study_uid in enumerate(study_uids):
    try:
        slot_slices, meta = prep_slots(
            study_uid,
            TRAIN_SERIES_DIR,
            series_df,
            n_groups=N_GROUPS,
            crop_mm=CROP_MM,
            out_size=OUT_SIZE,
            is_gold=study_uid in gold_uids,
        )
        save_study_npz(os.path.join(OUT_DIR, f'{study_uid}.npz'), slot_slices, meta)
        meta_rows.append(meta)
    except Exception as exc:
        # Broad on purpose: StudyDecodeError is the expected case, but a
        # malformed series_df row raises KeyError and a 320-slice series can
        # raise MemoryError, and neither should abort a multi-hour shard at
        # study 1,000 and lose every artifact with it.
        failed_studies.append({'StudyInstanceUID': study_uid,
                               'error': f'{type(exc).__name__}: {exc}'})
    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        print(f'{i + 1}/{len(study_uids)} prepped, {elapsed:.0f}s '
              f'({elapsed / (i + 1) * 1000:.0f} ms/study)')

elapsed = time.time() - t0
ms_per_study = elapsed / max(len(study_uids), 1) * 1000
print(f'done: {len(meta_rows)} prepped, {len(failed_studies)} failed, {elapsed:.0f}s')
for row in failed_studies[:10]:
    print(row)
assert meta_rows, 'no study prepped -- every gate below would report on an empty set'

In [ ]:
# GATE -- slot fill rate and why slots stayed empty. This is the number to know
# BEFORE spending GPU on the slot/attention screens: a slot that fills for a
# small minority means the attention head reads a mostly-masked input, and the
# screen could come back null for a data reason rather than a modelling one.
# Predicted from train_series.csv before the run: SAG_FLUID 94.2%, COR_FLUID
# 96.4%, AX_FLUID 100%, SAG_STRUCT 96.8%, COR_STRUCT 77.3%, AX_STRUCT 19.4%,
# mean 4.84 of 6. A large disagreement here means series on disk and series in
# the CSV do not match, which nothing downstream would notice.
slot_names = [name for name, _, _ in SLOTS]
fill = {name: 0 for name in slot_names}
reasons = {}
for m in meta_rows:
    for name in slot_names:
        if any(sm['slot'] == name for sm in m['series'].values()):
            fill[name] += 1
    for name, why in m['skipped_slots'].items():
        reasons[why] = reasons.get(why, 0) + 1

n = len(meta_rows)
print(f'{"slot":12s} {"filled":>7s}   rate')
for name in slot_names:
    print(f'{name:12s} {fill[name]:7d}  {fill[name] / n:6.1%}')
per_study = [len(m['series']) for m in meta_rows]
print(f'\nmean slots/study {np.mean(per_study):.2f} of {len(slot_names)}')
print('slots per study:', {k: per_study.count(k) for k in sorted(set(per_study))})
print('\nwhy a slot stayed empty:')
for why, count in sorted(reasons.items(), key=lambda kv: -kv[1]):
    print(f'  {count:6d}  {why}')

In [ ]:
# GATE -- physical scale is actually constant. The defect prep v2 exists to
# remove is invisible by construction: a study cropped at fixed pixels next to
# studies cropped at 130mm looks identical in the artifact and only shows up as
# a worse model. So assert it rather than eyeballing it.
series_meta_rows = []
for m in meta_rows:
    for series_uid, sm in m['series'].items():
        series_meta_rows.append({'StudyInstanceUID': m['StudyInstanceUID'],
                                 'SeriesInstanceUID': series_uid, **sm})
series_meta_df = pd.DataFrame(series_meta_rows)

mm_per_px = series_meta_df['mm_per_px'].unique()
print(f'distinct mm/px across {len(series_meta_df)} stored series: {mm_per_px}')
assert len(mm_per_px) == 1, 'physical scale is not constant -- the crop is broken'
print(f'PASS: every stored series at {mm_per_px[0]:.4f} mm/px')

# How much of each stored series was padding rather than anatomy. Predicted
# ~0.4% of the corpus has a field of view under the crop.
fov = series_meta_df[['fov_mm']].copy()
fov['min_fov'] = series_meta_df['fov_mm'].apply(min)
under = (fov['min_fov'] < CROP_MM).sum()
print(f'series with FOV under {CROP_MM:.0f}mm (zero-padded): {under} '
      f'({under / len(series_meta_df):.2%}), min FOV {fov["min_fov"].min():.0f}mm')
print(f'\nMB/study: ', end='')
sizes_mb = np.array([os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6
                     for f in os.listdir(OUT_DIR) if f.endswith('.npz')])
print(f'mean {sizes_mb.mean():.2f}, p50 {np.percentile(sizes_mb, 50):.2f}, '
      f'p95 {np.percentile(sizes_mb, 95):.2f}, max {sizes_mb.max():.2f} '
      f'-> shard total {sizes_mb.sum() / 1000:.2f} GB')

In [ ]:
# GATE -- read one artifact back and look at it. The two failure modes no
# counter catches: slot labels attached to the wrong plane, and slices within a
# group that are not actually adjacent.
from knee.prep import load_study_npz

t = time.time()
sample = [m['StudyInstanceUID'] for m in meta_rows][:100]
for uid in sample:
    slots, meta = load_study_npz(os.path.join(OUT_DIR, f'{uid}.npz'), order=slot_names)
print(f'load: {(time.time() - t) / len(sample) * 1000:.1f} ms/study (all slots, all slices)')

if SHARD_INDEX == 0:
    import matplotlib.pyplot as plt
    # Prefer studies that fill all six slots. AX_STRUCT is present for only
    # 19.4% of the corpus, so an arbitrary first-8 grid would very likely never
    # show it and the slot would ship unlooked-at.
    full = [m['StudyInstanceUID'] for m in meta_rows if len(m['series']) == len(slot_names)]
    rest = [m['StudyInstanceUID'] for m in meta_rows if m['StudyInstanceUID'] not in set(full)]
    show = (full[:5] + rest)[:8]
    # Look at this grid with one specific question, not just "is it a knee in
    # the right plane": crop_to_mm centres on the array, so if the joint sits
    # off-centre in the wide-FOV acquisitions the crop takes the wrong 130mm and
    # nothing else in this notebook would notice. Measured locally on 25 real
    # series: signal centroid a median 10.6mm from the array centre, max 26.5,
    # none past 30 (a quarter of the crop) -- but that is a proxy on a small
    # sample, and this grid is the real check.
    fig, axes = plt.subplots(len(show), len(slot_names),
                             figsize=(2 * len(slot_names), 2 * len(show)))
    for r, uid in enumerate(show):
        slots, meta = load_study_npz(os.path.join(OUT_DIR, f'{uid}.npz'), order=slot_names)
        for c, name in enumerate(slot_names):
            ax = axes[r, c]
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(name, fontsize=8)
            if name in slots:
                # middle slice of the middle group
                ax.imshow(slots[name][len(slots[name]) // 2], cmap='gray')
            else:
                ax.text(0.5, 0.5, 'empty', ha='center', va='center', fontsize=8)
        axes[r, 0].set_ylabel(meta['side'] or '?', fontsize=8)
    plt.tight_layout(); plt.show()

In [ ]:
# Manifest + the gold holdout list: is_gold lives in every meta, but a flat CSV
# is greppable without opening 4,407 archives, and the gold holdout is a hard
# requirement before any Phase 6 training.
manifest = pd.DataFrame([
    {
        'StudyInstanceUID': m['StudyInstanceUID'],
        'side': m['side'],
        'route': m['route'],
        'is_gold': m['is_gold'],
        'PatientSex': m['PatientSex'],
        'n_series_on_disk': m['n_series_on_disk'],
        'n_slots_filled': len(m['series']),
        'slots_filled': '|'.join(sorted(sm['slot'] for sm in m['series'].values())),
        'n_slices_stored': sum(sm['n_slices_stored'] for sm in m['series'].values()),
        'n_decode_failures': len(m['decode_failures']),
    }
    for m in meta_rows
])
manifest.to_csv(f'/kaggle/working/prep_manifest_shard{SHARD_INDEX}.csv', index=False)
series_meta_df.to_csv(f'/kaggle/working/prep_series_meta_shard{SHARD_INDEX}.csv', index=False)
pd.DataFrame(failed_studies, columns=['StudyInstanceUID', 'error']).to_csv(
    f'/kaggle/working/prep_failed_shard{SHARD_INDEX}.csv', index=False)

if SHARD_INDEX == 0:
    pd.DataFrame({'StudyInstanceUID': sorted(gold_uids)}).to_csv(
        '/kaggle/working/gold_study_uids.csv', index=False)

print(f'{len(os.listdir(OUT_DIR))} artifacts in {OUT_DIR}; manifests written')